In [86]:
import nilearn
from nilearn import datasets, plotting
from nilearn.connectome import ConnectivityMeasure
from nilearn.maskers import MultiNiftiLabelsMasker
import os
import requests
import csv
import pandas as pd
import nibabel as nib
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import scipy
from scipy.stats import multivariate_normal
from scipy.spatial.distance import pdist, squareform
from networkx.drawing.nx_agraph import graphviz_layout
from sklearn.metrics.cluster import mutual_info_score
from sklearn.feature_selection import mutual_info_regression
from scipy.sparse.csgraph import minimum_spanning_tree
from collections import deque, defaultdict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import GaussianNB
import collections
from collections import defaultdict, deque
import copy

In [87]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using device: {device}')

Using device: cpu


## **Load Data**

In [88]:
# set the working directory to fmri_connectivity_trees root directory
home_base_dir = '/Users/aj/dmello_lab/fmri_connectivity_trees' # directory where repository lives at home computer
lab_base_dir = '/Users/ajjain/Downloads/Code/fmri_connectivity_trees' # directory where repository lives at lab computer
utd_base_dir = '/mfs/io/groups/dmello/projects/dynamric/fmri_connectivity_trees'
biohpc_base_dir = '/project/greencenter/Lin_lab/s229618/fmri_connectivity_trees'

# set base directory depending on where the code is being run
base_dir = home_base_dir if os.path.exists(home_base_dir) else lab_base_dir
base_dir = utd_base_dir if os.path.exists(utd_base_dir) else base_dir
base_dir = biohpc_base_dir if os.path.exists(biohpc_base_dir) else base_dir


# get schaefer
# schaefer = datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=17, resolution_mm=1, data_dir=None, base_url=None, resume=True, verbose=1)
# schaefer_coords = plotting.find_parcellation_cut_coords(labels_img=schaefer.maps)

subjects = [
            'MSC01', 
            'MSC02',
            'MSC03',
            'MSC04',
            'MSC05', 
            'MSC06',
            'MSC07', 
            'MSC08',
            'MSC09',
            'MSC10'
            ]

sessions = [
    'func01', 
    'func02', 
    'func03', 
    'func04', 
    'func05', 
    'func06', 
    'func07', 
    'func08', 
    'func09', 
    'func10'
    ]
ids = ['rest']
# motor_ids = ['motor_run-01', 'motor_run-02']
atlas = 'glasser360_SUIT'
atlas_subdir = 'glasser360_SUIT'

# get glasser labels
file_path = f'{base_dir}/atlases/glasser360/glasser360NodeNames.txt'
with open(file_path, 'r') as file:
    glasser_labels = file.readlines()

temp = []
for label in glasser_labels:    
    label = label.strip()
    # reform to match region dataframe below
    components = label.split('_')
    temp.append(components[1] + '_' + components[0][0])
   
glasser_labels = temp

# get glasser region list
glasser_region_list = pd.read_csv(f'{base_dir}/atlases/glasser360/HCP-MMP1_UniqueRegionList.csv')


# get suit labels
suit_labels = pd.read_csv(f'{base_dir}/atlases/SUIT/atl-Anatom.tsv', sep='\t')
suit_table = pd.read_csv(f'{base_dir}/atlases/SUIT/atl-Anatom.csv', sep='\t')

# # get Thalamus labels
# thalamus_labels = pd.read_csv(f'{base_dir}/atlases/Thalamus/ThalamusProbs.MNIsymSpace.names.txt', sep=',', header=None)[1]

# # get Brainstem labels
# brainstem_labels = pd.read_csv(f'{base_dir}/atlases/Brainstem/BrainstemProbs.MNIsymSpace.names.txt', sep=',', header=None)[1]


glasser_coords = []
for row in glasser_region_list.iterrows():
    glasser_coords.append([row[1]['x-cog'], row[1]['y-cog'], row[1]['z-cog']])

# all labels
all_labels = glasser_labels + suit_labels['name'].to_list()[:-2]

glasser_cort_regions = np.unique(glasser_region_list['cortex'])

In [89]:
glasser_cort_regions

array(['Anterior_Cingulate_and_Medial_Prefrontal', 'Auditory_Association',
       'Dorsal_Stream_Visual', 'Dorsolateral_Prefrontal',
       'Early_Auditory', 'Early_Visual', 'Inferior_Frontal',
       'Inferior_Parietal', 'Insular_and_Frontal_Opercular',
       'Lateral_Temporal', 'MT+_Complex_and_Neighboring_Visual_Areas',
       'Medial_Temporal', 'Orbital_and_Polar_Frontal',
       'Paracentral_Lobular_and_Mid_Cingulate', 'Posterior_Cingulate',
       'Posterior_Opercular', 'Premotor', 'Primary_Visual',
       'Somatosensory_and_Motor', 'Superior_Parietal',
       'Temporo-Parieto-Occipital_Junction',
       'Temporo-Parieto_Occipital_Junction', 'Ventral_Stream_Visual'],
      dtype=object)

In [79]:
# thalamus_regions.sort()
# thalamus_regions = [[region] for region in thalamus_regions]

# thalamus_regions
# with open(f'/Users/ajjain/Downloads/Code/fmri_connectivity_trees/atlases/MorelAtlasMNI152/thalamus_regions', 'w', newline='') as csvfile:
#     csv_writer = csv.writer(csvfile)
#     csv_writer.writerows(thalamus_regions)

In [90]:
thalamus_regions = []

with open(f'{base_dir}/atlases/MorelAtlasMNI152/thalamus_regions', 'r') as f:
    reader = csv.reader(f, delimiter='\t')
    for row in reader:
        thalamus_regions.append(row[0])

thalamus_regions

['left_AD',
 'left_AM',
 'left_AV',
 'left_CL',
 'left_CM',
 'left_CeM',
 'left_Hb',
 'left_LD',
 'left_LGNmc',
 'left_LGNpc',
 'left_LP',
 'left_Li',
 'left_MAX_VOLUME',
 'left_MDmc',
 'left_MDpc',
 'left_MGN',
 'left_MV',
 'left_Pf',
 'left_Po',
 'left_PuA',
 'left_PuI',
 'left_PuL',
 'left_PuM',
 'left_Pv',
 'left_RN',
 'left_SG',
 'left_STh',
 'left_VAmc',
 'left_VApc',
 'left_VLa',
 'left_VLpd',
 'left_VLpv',
 'left_VM',
 'left_VPI',
 'left_VPLa',
 'left_VPLp',
 'left_VPM',
 'left_global',
 'left_mtt',
 'left_sPf',
 'left_thalamus_body',
 'right_AD',
 'right_AM',
 'right_AV',
 'right_CL',
 'right_CM',
 'right_CeM',
 'right_Hb',
 'right_LD',
 'right_LGNmc',
 'right_LGNpc',
 'right_LP',
 'right_Li',
 'right_MAX_VOLUME',
 'right_MDmc',
 'right_MDpc',
 'right_MGN',
 'right_MV',
 'right_Pf',
 'right_Po',
 'right_PuA',
 'right_PuI',
 'right_PuL',
 'right_PuM',
 'right_RN',
 'right_SG',
 'right_STh',
 'right_VAmc',
 'right_VApc',
 'right_VLa',
 'right_VLpd',
 'right_VLpv',
 'right_VM',
 

In [91]:
def load_data(subjects=subjects, measure="mutual_information", task="rest", atlas="Schaefer", atlas_subdir="schaefer_100", skl=False, num_bins=100, other_suffix=''):
    """
    Save the covariance matrix to a CSV file.
    """
    data = {}
    for subject in subjects:
        data_path = f'{base_dir}/code/functional_connectivity/midnight_scan_club/output/{measure}/{subject}/{atlas_subdir}'
        if skl:
            file_name = f'{task}_skl_{num_bins}bins'
        else:
            file_name = f'{task}_{num_bins}bins'
        if other_suffix != '':
            file_name += f'{other_suffix}'

        data[subject] = torch.from_numpy(np.load(f'{data_path}/{file_name}.npy'))
    return data

In [94]:
subjects = [
            'MSC01', 
            # 'MSC02',
            # 'MSC03',
            # 'MSC04',
            # 'MSC05', 
            # 'MSC06',
            # 'MSC07', 
            # 'MSC08',
            # 'MSC09',
            # 'MSC10'
            ]

task = 'rest'
num_bins = 100
other_suffix = 'func01-07'
atlas = 'glasser360_SUIT_Thalamus'
atlas_subdir = 'glasser360_SUIT_Thalamus'

mi = {}
mi_joint_probs = {}
product_of_marginals = {}
entropies = {}

mi[num_bins] = load_data(subjects=subjects, measure="mutual_information", task=task, atlas=atlas, atlas_subdir=atlas_subdir, skl=False, num_bins=num_bins, other_suffix=other_suffix)
mi_joint_probs[num_bins] = load_data(subjects=subjects, measure="joint_probs", task=task, atlas=atlas, atlas_subdir=atlas_subdir, skl=False, num_bins=num_bins, other_suffix=other_suffix)
product_of_marginals[num_bins] = load_data(subjects=subjects, measure="product_of_marginals", task=task, atlas=atlas, atlas_subdir=atlas_subdir, skl=False, num_bins=num_bins, other_suffix=other_suffix)
entropies[num_bins] = load_data(subjects=subjects, measure="entropies", task=task, atlas=atlas, atlas_subdir=atlas_subdir, skl=False, num_bins=num_bins, other_suffix=other_suffix)

In [96]:
mi[100]['MSC01'].shape

torch.Size([473, 473])

## **Print Matrices**